# Beta-Neutral Dynamic Portfolio

This notebook demonstrates `BetaScreenerStrategy`, which builds a dynamically-selected
**long/short equity portfolio** that targets near-zero market beta.

At each monthly rebalance:
1. Compute rolling 60-day OLS beta for 20 candidate stocks vs SPY
2. Select **5 lowest-beta** stocks as the long book
3. Select **5 highest-beta** stocks as the short book
4. Scale book sizes so that `Σ(w_i × β_i) ≈ 0`
5. BIL absorbs residual cash

**Universe**: 20 stocks spanning low-beta (JNJ, PG, KO…) to high-beta (NVDA, AMD, TSLA…)  
**Benchmark**: SPY (used for beta computation only, not traded)  
**Rebalance**: Monthly mid-month (`month_mid`)  
**Lookback**: 60 trading days

> ⚠️ **Margin required**: Short selling requires a margin/prime brokerage account. This is a simulation only.

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio.helpers.data import YFinance
from tiportfolio import (
    BetaScreenerStrategy, FixRatio, Schedule, ScheduleBasedEngine,
    compare_strategies, plot_strategy_comparison_interactive,
    rebalance_decisions_table,
)
from tiportfolio.report import plot_portfolio_with_assets_interactive
import numpy as np
import pandas as pd
import plotly.graph_objects as go

enable_data_source_cache("tiportfolio", cache_dir=".cache")

# 20-stock universe: low-beta, high-beta, mid-range
UNIVERSE = [
    # Low-beta (defensive)
    "JNJ", "PG", "KO", "WMT", "VZ", "ED", "MCD", "PEP",
    # High-beta (growth/speculative)
    "NVDA", "AMD", "META", "TSLA", "MELI", "PLTR", "COIN", "SMCI",
    # Mid-range (mega-cap tech)
    "MSFT", "AAPL", "AMZN", "GOOGL",
]
CASH      = "BIL"
BENCHMARK = "SPY"
N_LONG    = 5
N_SHORT   = 5
LOOKBACK  = 60
# COIN listed Apr 2021, PLTR listed Sep 2020 — use 2022 start for full history
START     = "2022-01-01"
END       = "2024-12-31"
INITIAL_VALUE = 10_000

print(f"Universe: {len(UNIVERSE)} stocks | N_LONG={N_LONG} N_SHORT={N_SHORT} LOOKBACK={LOOKBACK}")

Universe: 20 stocks | N_LONG=5 N_SHORT=5 LOOKBACK=60


In [2]:
yf = YFinance(auto_adjust=True)

all_fetch_symbols = UNIVERSE + [CASH, BENCHMARK]
df_all = yf.query(all_fetch_symbols, start_date=START, end_date=END)

prices = {}
for symbol in df_all["symbol"].unique():
    sub = df_all[df_all["symbol"] == symbol].set_index("date")[["open", "high", "low", "close"]]
    prices[symbol] = sub

print(f"Fetched {len(prices)} symbols")
for sym, d in prices.items():
    print(f"  {sym}: {len(d)} rows  {d.index[0].date()} → {d.index[-1].date()}")

Loaded cached bar data.

Fetched 22 symbols
  BIL: 752 rows  2022-01-03 → 2024-12-30
  SPY: 752 rows  2022-01-03 → 2024-12-30
  AMZN: 752 rows  2022-01-03 → 2024-12-30
  MCD: 752 rows  2022-01-03 → 2024-12-30
  ED: 752 rows  2022-01-03 → 2024-12-30
  AAPL: 752 rows  2022-01-03 → 2024-12-30
  NVDA: 752 rows  2022-01-03 → 2024-12-30
  PG: 752 rows  2022-01-03 → 2024-12-30
  MELI: 752 rows  2022-01-03 → 2024-12-30
  KO: 752 rows  2022-01-03 → 2024-12-30
  AMD: 752 rows  2022-01-03 → 2024-12-30
  VZ: 752 rows  2022-01-03 → 2024-12-30
  MSFT: 752 rows  2022-01-03 → 2024-12-30
  WMT: 752 rows  2022-01-03 → 2024-12-30
  JNJ: 752 rows  2022-01-03 → 2024-12-30
  TSLA: 752 rows  2022-01-03 → 2024-12-30
  GOOGL: 752 rows  2022-01-03 → 2024-12-30
  SMCI: 752 rows  2022-01-03 → 2024-12-30
  COIN: 752 rows  2022-01-03 → 2024-12-30
  PEP: 752 rows  2022-01-03 → 2024-12-30
  PLTR: 752 rows  2022-01-03 → 2024-12-30
  META: 752 rows  2022-01-03 → 2024-12-30


In [3]:
# Build benchmark DataFrame for BetaScreenerStrategy
spy_df = prices[BENCHMARK]

# prices_df for the engine: universe + BIL (21 symbols, not SPY)
engine_prices = {s: prices[s] for s in UNIVERSE + [CASH]}
print(f"Engine prices: {len(engine_prices)} symbols (universe + cash)")
print(f"SPY rows for benchmark: {len(spy_df)}")

Engine prices: 21 symbols (universe + cash)
SPY rows for benchmark: 752


In [4]:
strategy = BetaScreenerStrategy(
    universe=UNIVERSE,
    n_long=N_LONG,
    n_short=N_SHORT,
    cash_symbol=CASH,
    benchmark_prices=spy_df,
    lookback_days=LOOKBACK,
    book_size=0.5,
)

engine = ScheduleBasedEngine(
    allocation=strategy,
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)

result = engine.run(
    symbols=strategy.get_symbols(),
    start=START, end=END,
    prices_df=engine_prices,
)
print(result.summary())

/Users/tim/Development/TiPortfolio/src/tiportfolio/utils/beta_screener.py:137: UserWarning: BetaScreenerStrategy: insufficient history or benchmark data at 2022-01-03 00:00:00+00:00; using equal-weight fallback
  warnings.warn(
/Users/tim/Development/TiPortfolio/src/tiportfolio/utils/beta_screener.py:137: UserWarning: BetaScreenerStrategy: insufficient history or benchmark data at 2022-01-14 00:00:00+00:00; using equal-weight fallback
  warnings.warn(
/Users/tim/Development/TiPortfolio/src/tiportfolio/utils/beta_screener.py:137: UserWarning: BetaScreenerStrategy: insufficient history or benchmark data at 2022-02-15 00:00:00+00:00; using equal-weight fallback
  warnings.warn(
/Users/tim/Development/TiPortfolio/src/tiportfolio/utils/beta_screener.py:137: UserWarning: BetaScreenerStrategy: insufficient history or benchmark data at 2022-03-15 00:00:00+00:00; using equal-weight fallback
  warnings.warn(


Backtest Summary
----------------
Sharpe Ratio:        -0.5272
Sortino Ratio:       -0.8342
MAR Ratio:           -0.0724
CAGR:                -1.14%
Max Drawdown:        15.70%
Kelly Leverage:      -5.8604
Mean Excess Return:  -0.0474
Final Value:         9,664.12
Total Fee:           5.91
Rebalances:          36


In [5]:
fig = plot_portfolio_with_assets_interactive(result, mark_rebalances=True)
fig.show()

## Rolling Book Composition

Which stocks were selected as **Long** (low-beta) or **Short** (high-beta) at each rebalance.

In [6]:
# Build book-membership DataFrame from rebalance decisions
book_records = []
for d in result.rebalance_decisions:
    row = {"date": d.date.date()}
    for s in UNIVERSE:
        w = d.target_weights.get(s, 0.0)
        if w > 0.01:
            row[s] = "Long"
        elif w < -0.01:
            row[s] = "Short"
        else:
            row[s] = "-"
    book_records.append(row)

book_df = pd.DataFrame(book_records).set_index("date")

# Encode as numeric for heatmap: Long=1, Short=-1, Out=0
encode = {"Long": 1, "Short": -1, "-": 0}
book_num = book_df.replace(encode).astype(int)

fig_book = go.Figure(go.Heatmap(
    z=book_num.values,
    x=list(book_num.columns),
    y=[str(d) for d in book_num.index],
    colorscale=[[0, "#EF553B"], [0.5, "#EEEEEE"], [1, "#00CC96"]],
    zmin=-1, zmax=1,
    colorbar=dict(tickvals=[-1, 0, 1], ticktext=["Short", "Out", "Long"]),
    hoverongaps=False,
))
fig_book.update_layout(
    title="Rolling Book Composition (Long=green, Short=red, Out=grey)",
    xaxis_title="Stock",
    yaxis_title="Rebalance Date",
    height=max(300, 30 * len(book_num) + 80),
)
fig_book.show()

print(book_df.to_string())

             JNJ    PG    KO   WMT    VZ    ED   MCD   PEP   NVDA    AMD   META   TSLA   MELI   PLTR   COIN   SMCI   MSFT   AAPL   AMZN  GOOGL
date                                                                                                                                          
2022-01-14  Long  Long  Long  Long  Long     -     -     -      -      -      -      -      -      -      -  Short  Short  Short  Short  Short
2022-02-15  Long  Long  Long  Long  Long     -     -     -      -      -      -      -      -      -      -  Short  Short  Short  Short  Short
2022-03-15  Long  Long  Long  Long  Long     -     -     -      -      -      -      -      -      -      -  Short  Short  Short  Short  Short
2022-04-14  Long     -  Long  Long  Long  Long     -     -  Short  Short      -      -  Short  Short  Short      -      -      -      -      -
2022-05-16  Long     -  Long  Long  Long  Long     -     -  Short  Short      -      -  Short  Short  Short      -      -      -      -      -

## Portfolio Beta Over Time

Realized portfolio beta `Σ(w_i × β_i)` at each rebalance date. Should hover near 0.

In [7]:
# Build combined close-price DataFrame for beta computation
close_series = []
for sym in UNIVERSE:
    close_series.append(prices[sym]["close"].rename(sym))
prices_combined = pd.concat(close_series, axis=1).sort_index()

# Use epoch-seconds comparison to avoid tz/resolution mismatch between
# tz-naive YFinance datetimes and tz-aware engine rebalance dates
idx_epoch = prices_combined.index.astype("int64")  # seconds since epoch (datetime64[s] → int64)

beta_rows = []
for d in result.rebalance_decisions:
    cutoff_epoch = int(d.date.timestamp())
    hist = prices_combined[idx_epoch <= cutoff_epoch]
    betas = strategy._compute_betas(hist)
    if betas is not None:
        port_beta = sum(d.target_weights.get(s, 0.0) * betas[s] for s in UNIVERSE)
        beta_rows.append({"date": d.date.date(), "portfolio_beta": port_beta})

beta_df = pd.DataFrame(beta_rows)

fig_beta = go.Figure()
fig_beta.add_trace(go.Scatter(
    x=beta_df["date"],
    y=beta_df["portfolio_beta"],
    mode="lines+markers",
    name="Portfolio Beta",
    line=dict(color="#636EFA"),
))
fig_beta.add_hline(y=0, line_dash="dash", line_color="grey", annotation_text="β=0 target")
fig_beta.update_layout(
    title="Portfolio Beta Over Time",
    xaxis_title="Date",
    yaxis_title="Beta vs SPY",
    hovermode="x unified",
)
fig_beta.show()

print(f"Mean portfolio beta: {beta_df['portfolio_beta'].mean():.4f}")
print(f"Std  portfolio beta: {beta_df['portfolio_beta'].std():.4f}")

Mean portfolio beta: -0.1476
Std  portfolio beta: 0.1236


In [8]:
decisions_df = rebalance_decisions_table(result)
decisions_df

,date,equity_before,equity_after,fee_paid,JNJ_price,JNJ_qty_before,JNJ_trade_qty,JNJ_qty_after,JNJ_value_after,PG_price,...,GOOGL_price,GOOGL_qty_before,GOOGL_trade_qty,GOOGL_qty_after,GOOGL_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after
0,2022-01-14 00:00:00+00:00,10236.969,10236.929,0.040,148.496,6.589,0.305,6.894,1023.697,143.545,...,138.435,-6.949,-0.446,-7.395,-1023.697,77.937,128.323,3.026,131.349,10236.969
1,2022-02-15 00:00:00+00:00,10357.244,10357.099,0.145,148.028,6.894,0.103,6.997,1035.724,141.620,...,135.584,-7.395,-0.244,-7.639,-1035.724,77.929,131.349,1.558,132.907,10357.244
2,2022-03-15 00:00:00+00:00,10650.177,10650.121,0.056,156.840,6.997,-0.206,6.790,1065.018,135.705,...,128.229,-7.639,-0.667,-8.306,-1065.018,77.930,132.907,3.756,136.663,10650.177
3,2022-04-14 00:00:00+00:00,11032.242,11030.885,1.358,160.188,6.790,0.097,6.887,1103.224,143.201,...,125.780,-8.306,8.306,0.000,0.000,77.939,136.663,-51.733,84.930,6619.345
4,2022-05-16 00:00:00+00:00,11150.269,11150.196,0.073,158.568,6.887,0.145,7.032,1115.027,140.871,...,113.587,0.000,0.000,0.000,0.000,77.964,84.930,0.880,85.810,6690.161
5,2022-06-15 00:00:00+00:00,10799.285,10799.243,0.042,152.337,7.032,0.057,7.089,1079.929,120.338,...,108.941,0.000,0.000,0.000,0.000,77.982,85.810,-2.720,83.090,6479.571
6,2022-07-15 00:00:00+00:00,11116.335,11116.206,0.129,159.721,7.089,-0.129,6.960,1111.634,131.744,...,110.939,0.000,0.000,0.000,0.000,78.019,83.090,2.399,85.489,6669.801
7,2022-08-15 00:00:00+00:00,10675.307,10675.082,0.226,148.842,6.960,0.212,7.172,1067.531,135.789,...,121.165,0.000,0.000,0.000,0.000,78.119,85.489,-3.497,81.993,6405.184
8,2022-09-15 00:00:00+00:00,10579.985,10579.840,0.145,148.931,7.172,-0.068,7.104,1057.999,125.625,...,102.138,0.000,0.000,0.000,0.000,78.298,81.993,-0.918,81.075,6347.991
9,2022-10-14 00:00:00+00:00,10354.919,10354.533,0.386,148.371,7.104,-0.125,6.979,1035.492,114.328,...,95.836,0.000,0.000,0.000,0.000,78.430,81.075,0.349,81.424,6386.111


## Comparison to Baselines

- **Long SPY**: 100% SPY buy-and-hold — pure market beta = 1
- **Equal-weight universe**: equal allocation across all 20 universe stocks — beta ≈ market-weight

In [9]:
# Baseline 1: 100% long SPY
engine_spy = ScheduleBasedEngine(
    allocation=FixRatio(weights={BENCHMARK: 1.0}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_spy = engine_spy.run(
    symbols=[BENCHMARK], start=START, end=END,
    prices_df={BENCHMARK: prices[BENCHMARK]},
)
print("Long SPY")
print(result_spy.summary())

Long SPY
Backtest Summary
----------------
Sharpe Ratio:        0.3421
Sortino Ratio:       0.4848
MAR Ratio:           0.3590
CAGR:                8.80%
Max Drawdown:        24.50%
Kelly Leverage:      1.9526
Mean Excess Return:  0.0599
Final Value:         12,866.22
Total Fee:           0.00
Rebalances:          0


In [10]:
# Baseline 2: equal weight across universe (no cash, no shorts)
eq_weights = {s: 1.0 / len(UNIVERSE) for s in UNIVERSE}
engine_eq = ScheduleBasedEngine(
    allocation=FixRatio(weights=eq_weights),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_eq = engine_eq.run(
    symbols=UNIVERSE, start=START, end=END,
    prices_df={s: prices[s] for s in UNIVERSE},
)
print("Equal-weight universe")
print(result_eq.summary())

Equal-weight universe
Backtest Summary
----------------
Sharpe Ratio:        0.8965
Sortino Ratio:       1.3757
MAR Ratio:           0.8387
CAGR:                25.54%
Max Drawdown:        30.45%
Kelly Leverage:      3.6910
Mean Excess Return:  0.2177
Final Value:         19,738.43
Total Fee:           2.48
Rebalances:          36


In [11]:
compare_strategies(
    result, result_spy, result_eq,
    names=["BetaNeutral", "Long SPY", "Equal-weight Universe"],
)

,BetaNeutral,Long SPY,Equal-weight Universe,better
metric,,,,
sharpe_ratio,-0.527205,0.342053,0.896454,Equal-weight Universe
sortino_ratio,-0.834239,0.484783,1.375716,Equal-weight Universe
mar_ratio,-0.072393,0.359033,0.838750,Equal-weight Universe
cagr,-0.011362,0.087950,0.255381,Equal-weight Universe
max_drawdown,0.156956,0.244964,0.304478,BetaNeutral


In [12]:
plot_strategy_comparison_interactive(result, result_spy, result_eq)

## Interpretation

### Beta Neutrality
By holding low-beta stocks long and high-beta stocks short — sized so that
`Σ(w_long × β_long) + Σ(w_short × β_short) ≈ 0` — the portfolio decouples P&L from overall
market direction. Profits when low-beta stocks **outperform** high-beta stocks on a risk-adjusted basis.

### Dynamic Selection
Unlike a fixed long/short pair, this strategy re-screens the universe every month. As individual
stock betas shift (e.g. TSLA beta rising in a risk-off environment), the composition adapts.
The book-composition heatmap shows how frequently stocks rotate in and out.

### Monthly Freeze Period
Using `Schedule("month_mid")` limits rebalances to once per month. Between rebalances, weights
drift with prices but no new trades are made. This avoids over-trading on daily beta fluctuations
and controls transaction costs.

### Short Book Sizing
The short book is scaled as:
```
short_book_size = book_size × avg_beta_long / avg_beta_short   (clamped 0.1–2.0)
```
This ensures the dollar-weighted beta cancels. When high-beta stocks have very high betas,
fewer short dollars are needed to offset the long book.

---

> ⚠️ **Margin account required**: Short positions require posting margin collateral with a
> prime broker or margin-enabled brokerage. BIL represents the T-bill collateral earning near
> risk-free yield while short positions are open.
>
> This notebook is for educational/research purposes. Past performance does not guarantee future results.